# Course-End Project 1: AI-Powered HR Assistant

## Objective
Build a simple conversational HR assistant that answers questions from the supplied Nestlé Human Resources Policy PDF.

### Workflow
PDF → Text Chunks → Embeddings → FAISS → GPT-3.5 Turbo → Gradio Chatbot

The original project specifies PyPDFLoader, text splitting, ChromaDB, OpenAI embeddings, GPT-3.5 Turbo, a prompt template, and Gradio. Because ChromaDB causes SQLite compatibility problems in the target lab, FAISS is used instead.

## 1. Import Libraries

In [1]:
import openai
import gradio as gr

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

## 2. Set API Key

In [2]:
import os
from dotenv import load_dotenv

ENV_PATH = os.path.expanduser("~/Documents/AIML/GitAIML/Full-Stack-Data-Science-WIth-Gen-AI-and-Agentic-AI/GenAI/.env")
load_dotenv(ENV_PATH, override=True)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(f"OPENAI_API_KEY not found in {ENV_PATH}")
openai.api_key = os.getenv("OPENAI_API_KEY")

print("API key loaded")

API key loaded


## 3. Load the HR Policy PDF

In [3]:
PDF_PATH = "the_nestle_hr_policy_pdf_2012.pdf"

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print("Pages loaded:", len(documents))

Pages loaded: 8


## 4. Split the Document

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)
print("Chunks created:", len(chunks))

Chunks created: 26


## 5. Create Embeddings and FAISS Vector Store

In [5]:
embeddings = OpenAIEmbeddings()
db = FAISS.from_documents(chunks, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 3})

## 6. Create the QA Chain

In [6]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""Answer the question using only the HR policy context.
If the answer is not in the context, say: Information not found in the policy.

Context:
{context}

Question:
{question}

Answer:"""
)

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# "Stuff" retrieval QA chain: retrieve chunks, stuff them into the prompt, ask the LLM
qa = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 7. Test the Assistant

In [7]:
question = "What are the key elements of Total Rewards?"
print(qa.invoke(question))

Fixed Pay, Variable Pay, Benefits, Personal Growth and Development, and Work Life Environment.


## 8. Gradio Chatbot

In [8]:
def ask_hr(question):
    if not question.strip():
        return "Please enter a question."
    return qa.invoke(question)

demo = gr.Interface(
    fn=ask_hr,
    inputs=gr.Textbox(label="Ask an HR question"),
    outputs=gr.Textbox(label="Answer"),
    title="Nestlé HR Policy Assistant",
    description="Ask questions about the supplied Nestlé HR Policy PDF."
)

demo.launch(
    server_name="0.0.0.0",
    share=True,
    inline=True
)

* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://f47f8f7243990cc7b8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Example Questions
- What is the role of line managers?
- What are the key elements of Total Rewards?
- What does the policy say about training and learning?
- How are promotions described in the policy?
- What does the policy say about employee relations?